# R for Biostatistics, Ecology, and Genomics Workflow

This notebook scaffold mirrors the R-first workflow: biostatistical summaries, ecological diversity, genomics count normalization, metadata validation, and reproducibility documentation.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

article_dir = Path.cwd().parent
biostat = pd.read_csv(article_dir / 'data' / 'biostat_measurements.csv')
biostat[biostat['qc_flag'] == 'pass'].groupby('treatment')['response'].agg(['count', 'mean', 'std']).round(5)

In [ ]:
eco = pd.read_csv(article_dir / 'data' / 'ecology_counts.csv')
def shannon(x):
    x = np.array([v for v in x if v > 0], dtype=float)
    p = x / x.sum()
    return float(-(p * np.log(p)).sum())

eco_summary = eco.groupby(['site', 'habitat']).agg(total_abundance=('count', 'sum'), richness=('count', lambda x: (x > 0).sum()), shannon=('count', shannon)).reset_index()
eco_summary.round(5)

In [ ]:
counts = pd.read_csv(article_dir / 'data' / 'genomics_counts.csv').set_index('gene_id')
metadata = pd.read_csv(article_dir / 'data' / 'genomics_metadata.csv')
counts = counts[metadata['sample_id']]
library_sizes = counts.sum(axis=0)
cpm = counts.divide(library_sizes, axis=1) * 1_000_000
control = metadata.loc[metadata['condition'] == 'control', 'sample_id']
treated = metadata.loc[metadata['condition'] == 'treated', 'sample_id']
summary = pd.DataFrame({'gene_id': counts.index, 'mean_cpm_control': cpm[control].mean(axis=1), 'mean_cpm_treated': cpm[treated].mean(axis=1)})
summary['log2_fold_change'] = np.log2((summary['mean_cpm_treated'] + 1) / (summary['mean_cpm_control'] + 1))
summary.round(5)